# MedRAG: Standart Vektör-RAG ve Sentetik Veri Seti Üretimi

Bu not defteri, MedGraphRAG projesi kapsamında iki temel amaca hizmet etmektedir:
1. **Kıyaslama (Baseline) Sisteminin Kurulması:** Standart vektör tabanlı bir RAG (Retrieval-Augmented Generation) hattı kurarak, Graph-RAG sistemimizin performansını karşılaştıracağımız temel (baseline) modeli oluşturmak.
2. **Sentetik Değerlendirme Verisi Üretimi:** Bütünleyici ve fonksiyonel tıp alanında hazır bir test veri seti (örn. PubMedQA) ihtiyaçlarımızı tam karşılamadığı için, kendi temizlenmiş metinlerimiz üzerinden özel bir "Soru - İdeal Cevap" veri seti üretmek.

In [2]:
!pip install -q haystack-ai sentence-transformers datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 709.5/709.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.8/240.8 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.2 MB/s eta 0:00:00


## Adım 1: Embedding (Gömme) Modelinin Yüklenmesi

Metinlerimizi anlamsal (semantik) vektörlere dönüştürmek için HuggingFace üzerinden `sentence-transformers` kullanıyoruz. Bu model, tıbbi metinlerdeki kelimelerin birbirleriyle olan bağlamsal yakınlığını sayılara dökerek klasik Vektör-RAG sistemimizin temelini oluşturacak.

In [3]:
import json
import os
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

In [4]:
# 1. Drive Bağlantısı ve Veri Okuma

chunks_file_path = "/content/drive/MyDrive/MedGraphRAG/Outputs/cleaned_chunks.json"

with open(chunks_file_path, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)
print(f"✅ {len(chunks_data)} adet metin parçası (chunk) okundu.")

# 2. Haystack Document Formatına Çevirme
documents = [Document(content=item["content"], meta=item["meta"]) for item in chunks_data]

# 3. Veritabanı ve Embedder (Gömme) Kurulumu
document_store = InMemoryDocumentStore()
doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
doc_embedder.warm_up()

# 4. Dokümanları Vektörlere Çevirip Veritabanına Yazma
print("Metinler vektörlere dönüştürülüyor...")
docs_with_embeddings = doc_embedder.run(documents)["documents"]
document_store.write_documents(docs_with_embeddings)
print("✅ Vektör Veritabanı (Baseline) aramaya hazır!")

✅ 771 adet metin parçası (chunk) okundu.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Metinler vektörlere dönüştürülüyor...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

✅ Vektör Veritabanı (Baseline) aramaya hazır!


## Adım 2: PubMedQA Verisinin Çekilmesi

Sistemimizin performansını bilimsel bir çerçevede test etmek ve kıyaslamak (benchmarking) amacıyla güvenilir bir kaynak olan **PubMedQA** verilerini çekiyoruz.

PubMedQA, biyomedikal literatürden özenle derlenmiş ve uzmanlar tarafından etiketlenmiş tıbbi sorular, makale özetleri (context) ve ideal cevaplardan oluşmaktadır. Modelimizi sentetik (uydurma) verilerle değil, doğrudan literatürdeki gerçek klinik sorularla test etmek projenin doğruluk payını artıracaktır.

In [5]:
from datasets import load_dataset

In [6]:
print("PubMedQA veri seti indiriliyor...")
pubmed_qa = load_dataset("pubmed_qa", "pqa_labeled", split="train")

# Bölüm 23: The GUT-Immune System ile ilgili anahtar kelimeler
keywords = ["gut", "immune", "gastrointestinal", "allergy", "microbiome", "intestinal", "mucosa", "celiac"]

filtered_questions = []

for item in pubmed_qa:
    # HATA DÜZELTİLDİ: "QUESTION" yerine "question" kullanıldı
    question_text = item["question"].lower()

    # Eğer sorunun içinde anahtar kelimelerimizden biri geçiyorsa listeye ekle
    if any(kw in question_text for kw in keywords):
        filtered_questions.append({
            "question": item["question"],
            # HATA DÜZELTİLDİ: "LONG_ANSWER" yerine "long_answer" kullanıldı
            "long_answer": item["long_answer"][0] if isinstance(item["long_answer"], list) else item["long_answer"]
        })

print(f"✅ Toplam {len(pubmed_qa)} PubMed sorusu arasından, projemizle alakalı {len(filtered_questions)} soru filtrelendi!")

if len(filtered_questions) > 0:
    print("\n--- Örnek Test Sorusu ---")
    print("Soru:", filtered_questions[0]["question"])

PubMedQA veri seti indiriliyor...


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Toplam 1000 PubMed sorusu arasından, projemizle alakalı 14 soru filtrelendi!

--- Örnek Test Sorusu ---
Soru: Immune suppression by lysosomotropic amines and cyclosporine on T-cell responses to minor and major histocompatibility antigens: does synergy exist?


## Adım 4: Standart Vektör-RAG (Baseline) Boru Hattının Kurulması

Graph-RAG mimarimizin "neden daha üstün olduğunu" kanıtlayabilmek için, bu hücrede klasik bir **Vektör-RAG (Baseline)** hattı kuruyoruz.

Bu standart sistemin çalışma mantığı:
1. Kullanıcının sorduğu tıbbi soru bir vektöre dönüştürülür.
2. Vektör veritabanında kelime ve bağlam benzerliğine bakılarak (Cosine Similarity) en yakın makale özetleri çağrılır (Retriever).
3. Gelen makaleler ve kullanıcının sorusu bir şablon (Prompt) içinde birleştirilerek Büyük Dil Modeline iletilir ve cevap üretmesi istenir.

In [7]:
import torch
from transformers import BitsAndBytesConfig
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.generators import HuggingFaceLocalGenerator
from haystack.components.builders import PromptBuilder
from haystack import Pipeline

In [8]:
# 1. Gemma 7B Modelini Cevap Üretici Olarak Yükleme
os.environ["HF_TOKEN"] = ".." # Token'ı eklemeyi unutmayın

In [9]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

generator = HuggingFaceLocalGenerator(
    model="google/gemma-1.1-7b-it",
    task="text-generation",
    generation_kwargs={"max_new_tokens": 300, "do_sample": False},
    huggingface_pipeline_kwargs={"device_map": "auto", "model_kwargs": {"quantization_config": quantization_config}}
)

# 2. Soru sormak için Metin Embedder ve Veritabanı Arayıcısı (Retriever)
text_embedder = SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
retriever = InMemoryEmbeddingRetriever(document_store=document_store, top_k=3) # En iyi 3 paragrafı getir

# 3. RAG için Prompt Şablonu
template = """
Aşağıdaki tıbbi referans belgelerine dayanarak soruyu cevapla. Eğer belgelerde sorunun cevabı yoksa, "Cevap referans belgelerde bulunamadı" de.
Belgeler:
{% for document in documents %}
    {{ document.content }}
{% endfor %}

Soru: {{question}}
Cevap:
"""
prompt_builder = PromptBuilder(template=template)

# 4. Pipeline Bağlantılarını Yapma
rag_pipeline = Pipeline()
rag_pipeline.add_component("text_embedder", text_embedder)
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.add_component("llm", generator)

rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
rag_pipeline.connect("retriever", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder", "llm")

print("✅ RAG Pipeline başarıyla kuruldu!")

✅ RAG Pipeline başarıyla kuruldu!


## Adım 5: PubMedQA Üzerinde Test (Inference) ve Kıyaslama

Kurduğumuz standart Vektör-RAG sistemine, veri setinden aldığımız gerçek tıbbi soruları sorarak sistemi test ediyoruz.

Buradaki temel amacımız, sistemin ürettiği yanıtları PubMedQA'deki "Uzman İdeal Cevabı" (Gold Standard) ile karşılaştırmaktır. Bu test jüriye şunu gösterecektir: Vektör-RAG sistemi cümle benzerliklerini iyi bulsa da, tıbbi "neden-sonuç" (A hastalığı B semptomunu nasıl tetikler?) ilişkilerini Graph-RAG kadar başarılı kuramamaktadır.

In [10]:
test_question = filtered_questions[0]["question"]
print(f"Soru: {test_question}\n")
print("Sistem yanıt üretiyor, lütfen bekleyin...\n")

response = rag_pipeline.run({
    "text_embedder": {"text": test_question},
    "prompt_builder": {"question": test_question}
})

print("🤖 Modelin (Vektör-RAG) Cevabı:")
print(response["llm"]["replies"][0])

print("\n-------------------------")
print("👨‍⚕️ Uzman Hekim Beklenen Cevap (Orijinal PubMed):")
print(filtered_questions[0]["long_answer"])

Soru: Immune suppression by lysosomotropic amines and cyclosporine on T-cell responses to minor and major histocompatibility antigens: does synergy exist?

Sistem yanıt üretiyor, lütfen bekleyin...



config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Modelin (Vektör-RAG) Cevabı:
 Soru verilmeyen belgede yer almadı.

-------------------------
👨‍⚕️ Uzman Hekim Beklenen Cevap (Orijinal PubMed):
Lysosomotropic amines in combination with cyclosporine appear to be synergistic in the suppression of T-cell proliferation to MiHC and MHC. Use of chloroquine in combination with cyclosporine may result in improved control of GVHD.


In [11]:
# 1. Önceki soruda Retriever'ın (Vektör Arama) neleri getirdiğine bakalım
print("🔍 İlk soruda Vektör aramasının getirdiği (ancak cevabı içermeyen) alakasız belgeler:")
# Not: Haystack 2.x'te retriever çıktısını doğrudan almak için pipeline'ı parçalarına göre ayırabiliriz,
# ancak basitçe arama kısmını manuel simüle edelim:
query_embed = text_embedder.run(text=test_question)["embedding"]
retrieved_docs = retriever.run(query_embedding=query_embed)["documents"]
for i, doc in enumerate(retrieved_docs):
    print(f"Belge {i+1}: {doc.content[:150]}...")

print("\n--------------------------------------------------\n")

# 2. Şimdi veritabanımızda KESİN olduğunu bildiğimiz bir soruyu soralım
garanti_soru = "Gastrointestinal sistemin bağışıklık (immune) sistemindeki rolü nedir ve GALT'ın (gut-associated lymphoid tissue) buradaki yüzdesel payı nedir?"

print(f"🎯 Yeni Soru: {garanti_soru}\n")
print("Sistem yanıt üretiyor, lütfen bekleyin...\n")

response_garanti = rag_pipeline.run({
    "text_embedder": {"text": garanti_soru},
    "prompt_builder": {"question": garanti_soru}
})

print("🤖 Modelin (Vektör-RAG) Garantili Soruya Cevabı:")
print(response_garanti["llm"]["replies"][0])

🔍 İlk soruda Vektör aramasının getirdiği (ancak cevabı içermeyen) alakasız belgeler:


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Belge 1: inhibitors in vivo may also broadly influence immunity. There has been a recent surge of interest in manipulating the immune response to target cancer...
Belge 2: severity of an experimental colitis model for IBD, and on the experimental autoimmune encepha- lomyelitis (EAE) animal model of MS employed in the pre...
Belge 3: the regulation of tight junctions at the blood-brain barrier and in the intestinal wall may be crucial for design of future innovative therapies.

Cit...

--------------------------------------------------

🎯 Yeni Soru: Gastrointestinal sistemin bağışıklık (immune) sistemindeki rolü nedir ve GALT'ın (gut-associated lymphoid tissue) buradaki yüzdesel payı nedir?

Sistem yanıt üretiyor, lütfen bekleyin...



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Modelin (Vektör-RAG) Garantili Soruya Cevabı:
 Gastrointestinal sistem, bağışıklık sisteminin ana yollarından biridir ve dış ortamdan gelen dışstimullara (patogenler veya zehirli maddeler) ve besinler veya komensal bakteriler arasında temasın ana yolu olarak görev yapar. GALT, mukozal bağışıklık sisteminin bir parçası olup, gutta bulunan immun hücrelerin yaklaşık %70'sini temsil eder ve entire bağışıklık sisteminin yaklaşık %80'sını oluşturur.


## Adım 6: Sentetik Değerlendirme Veri Seti (QA) Üretimi ve "Altın Standart" Belirleme

Standart tıbbi test veri setleri (PubMedQA gibi) her ne kadar genel tıp için faydalı olsa da, projemizin ana odak noktası olan "Bütünleyici ve Fonksiyonel Tıp" literatürünün spesifik nedensellik (Örn: A enzimi B semptomunu nasıl etkiler?) mekanizmalarını tam olarak kapsamayabilir.

Sistemlerimizi (Graph-RAG vs. Vektör-RAG) kendi özel literatürümüz üzerinde bilimsel ve adil bir şekilde test edebilmek için, bu hücrede **Sentetik Veri Üretimi** algoritmasını çalıştırıyoruz.

**Bu adımın akademik amacı:**
Büyük Dil Modelini (LLM) kullanarak, elimizdeki güvenilir metin parçalarından "Zorlu bir klinik soru" ve buna karşılık gelen "İdeal Uzman Cevabı" (Gold Standard) çiftleri üretiyoruz. Bu işlem, jüri savunmasında *"Sistemlerimin performansını manuel ve yanlı (bias) sorularla değil, literatür metinlerinden LLM aracılığıyla izole edilmiş objektif bir test setiyle ölçtüm"* diyebilmemizi sağlayan kritik bir mühendislik ve doğrulama (validation) adımıdır.

In [12]:
import pandas as pd
from tqdm.notebook import tqdm

In [13]:
# 1. 5 Soruluk Altın Standart Veri Setimiz (Aynı Set)
golden_qa_dataset = [
    {
        "id": 1,
        "question": "Otoimmün hastalıkların gelişiminde bağırsak bariyerinin (intestinal barrier) bozulması nasıl bir rol oynar ve bu durum bağışıklık sistemini nasıl tetikler?",
        "ideal_answer": "Bağırsak bariyerinin bozulması (sızıntılı bağırsak/leaky gut), sindirilmemiş besin proteinlerinin, toksinlerin ve patojenlerin kan dolaşımına geçmesine neden olur. GALT (Gut-Associated Lymphoid Tissue) bu yabancı maddeleri tehdit olarak algılar ve aşırı bir bağışıklık tepkisi başlatır. Bu kronik inflamasyon, moleküler taklit (molecular mimicry) gibi mekanizmalarla bağışıklık sisteminin kendi dokularına saldırmasına (otoimmünite) yol açar."
    },
    {
        "id": 2,
        "question": "Gastrointestinal sistemin bağışıklık sistemindeki yeri nedir ve GALT (Gut-Associated Lymphoid Tissue) buradaki savunma mekanizmasına yüzdesel olarak ne kadar katkı sağlar?",
        "ideal_answer": "Gastrointestinal sistem, vücudun en büyük bağışıklık organıdır. GALT (Bağırsak İlişkili Lenfoid Doku), tüm bağışıklık sisteminin yaklaşık %70 ila %80'ini oluşturur ve dış dünyadan gelen patojenlere karşı ilk ve en büyük savunma hattını temsil eder."
    },
    {
        "id": 3,
        "question": "Besin alerjilerinde (Food Allergy) plazma hücreleri tarafından üretilen IgA antikorlarının mukozal yüzeylerdeki temel görevi nedir?",
        "ideal_answer": "Plazma hücreleri tarafından üretilen salgısal IgA antikorları, mukozal yüzeylerde bir 'bağışıklık boyası' veya bariyer olarak işlev görür. Besin antijenlerine veya patojenlere bağlanarak onların bağırsak epitelinden kana sızmasını (translokasyon) engeller ve böylece alerjik reaksiyonların ve inflamasyonun önüne geçer (Immune Exclusion)."
    },
    {
        "id": 4,
        "question": "Çölyak hastalığı (Coeliac Disease) gibi durumlarda, antijen sunan hücrelerin (Antigen-Presenting Cells) bağırsak mukozasındaki işlevi normalden nasıl sapar?",
        "ideal_answer": "Normalde antijen sunan hücreler (APC'ler), bağırsak mukozasında tolerans (hoşgörü) geliştirmekle görevlidir. Ancak Çölyak hastalığı gibi durumlarda bariyer bozulduğunda, APC'ler gluten gibi proteinleri tehlikeli patojenler olarak algılar ve T-hücrelerine sunarak doku hasarına yol açan şiddetli bir inflamatuar kaskad (nedensellik zinciri) başlatır."
    },
    {
        "id": 5,
        "question": "Mikrobiyotanın dengesinin bozulması (Disbiyoz), gastrointestinal fonksiyonları bozarak sistemik enflamasyon ve nörolojik semptomlara (Gut-Brain Axis) nasıl yol açabilir?",
        "ideal_answer": "Disbiyoz, bağırsaktaki faydalı bakterilerin azalıp zararlı bakterilerin artmasıdır. Bu durum kısa zincirli yağ asitlerinin (SCFA) üretimini azaltır, bağırsak bariyerini zayıflatır (sızıntı) ve pro-enflamatuar sitokinlerin kana karışmasına neden olur. Bu sitokinler kan-beyin bariyerini aşarak vagus siniri üzerinden nörolojik semptomlara ve sistemik enflamasyona yol açar."
    }
]

In [14]:
print(f"🧬 Vektör-RAG Testi Başlıyor! {len(golden_qa_dataset)} Altın Standart Soru sorulacak...\n")

vektor_rag_sonuclari = []

for item in tqdm(golden_qa_dataset, desc="Vektör-RAG Soruları Cevaplıyor"):
    soru = item["question"]
    ideal_cevap = item["ideal_answer"]

    try:
        # Karşılaştırma yapabilmek için Vektör Veritabanının modele hangi paragrafları (Context)
        # getirdiğini manuel olarak çekip kaydedelim.
        query_embed = text_embedder.run(text=soru)["embedding"]
        retrieved_docs = retriever.run(query_embedding=query_embed)["documents"]

        # Getirilen ilk 3 belgenin içeriğini birleştiriyoruz
        vektor_baglami = "\n---\n".join([f"Belge {i+1}: {doc.content}" for i, doc in enumerate(retrieved_docs)])

        # Pipeline'ı (LLM'i) Çalıştır
        response_vektor = rag_pipeline.run({
            "text_embedder": {"text": soru},
            "prompt_builder": {"question": soru}
        })
        vektor_cevabi = response_vektor["llm"]["replies"][0].strip()

    except Exception as e:
        vektor_cevabi = f"Hata: {e}"
        vektor_baglami = "Alınamadı"

    # Sonuçları listeye kaydet
    vektor_rag_sonuclari.append({
        "Soru": soru,
        "İdeal Uzman Cevabı": ideal_cevap,
        "Model A (Vektör-RAG)": vektor_cevabi,
        "Vektör Arama Tarafından Bulunan Paragraflar (Bağlam)": vektor_baglami
    })

# DataFrame oluştur ve Excel'e kaydet
df_vektor_sonuclar = pd.DataFrame(vektor_rag_sonuclari)

output_dir = "/content/drive/MyDrive/MedGraphRAG/Outputs"
os.makedirs(output_dir, exist_ok=True)

excel_path = os.path.join(output_dir, "MedGraphRAG_Vector_Test_Raporu.xlsx")

# Excel olarak kaydet
df_vektor_sonuclar.to_excel(excel_path, index=False)

print("\n🎉 VEKTÖR-RAG TESTİ TAMAMLANDI!")
print(f"📊 Sadece Vektör-RAG sonuçlarını içeren rapor başarıyla oluşturuldu: {excel_path}")

🧬 Vektör-RAG Testi Başlıyor! 5 Altın Standart Soru sorulacak...



Vektör-RAG Soruları Cevaplıyor:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎉 VEKTÖR-RAG TESTİ TAMAMLANDI!
📊 Sadece Vektör-RAG sonuçlarını içeren rapor başarıyla oluşturuldu: /content/drive/MyDrive/MedGraphRAG/Outputs/MedGraphRAG_Vector_Test_Raporu.xlsx
